# M07b.3/.4: final-eval no-regression, re-probe, then freeze D1

**Thin launcher.** The logic, locks and gates live in the repo:

    python -m pii_redteam.cli freeze-d1 --config configs/m07b_freeze_d1.yaml

**Runtime -> Change runtime type -> T4 GPU, then Run all.** It needs on Drive: the D1
training run `pii-redteam-rl/d1-20261108-145253/` (with `d1-candidate/`), frozen D0 at
`pii-redteam-rl/d0-20261004-142714/d0-best`, and the dataset cache. The probe also
downloads Qwen/Qwen3.5-2B.

In order, stopping at the first failed lock:

1. **Locks:** final-eval partition sha, D0 sha, D1 candidate sha (vs the training report
   and the config), the training and M07a configs unchanged, repo commits recorded, and
   no existing frozen directory.
2. **Final-eval (single use):** D0 and D1 on the held-out AI4Privacy partition. Gates:
   span-F1 drop <= 0.01, the pre-registered precision/recall drop <= 0.01 from the
   training config, and a **proposed** precision guard on name-free rows (D1's flagged-row
   rate at most +25% relative to D0's). If the training run's
   `d1_final_eval_report.json` already exists, it is reused, never re-scored.
3. **Short negatives (optional):** `data/synthetic/short_eval_neg_v1.jsonl`, if present and
   pinned. Gated on place / org_product / calendar / role_only; homograph rows are
   reported D0 vs D1 only. Absent -> recorded as "skipped: not provided".
4. **M07a probe against D1:** the unchanged M07a probe (same bank, prompt, thresholds)
   with D1 as the scorer; D0 is rescored on the same texts. D1 must pass the go/no-go
   table with its miss rate inside the 1-30% band. The timing pilot prints the
   projection first (3,400 generations).
5. **Freeze:** only if every required check passes, D1 is copied to
   `pii-redteam-rl/d1-frozen-d1-20261108-145253/` with `freeze_manifest.json`.
   Otherwise the status is `freeze_blocked` with the failing checks. An existing frozen
   directory is never overwritten.

A blocked freeze is a recorded result, not a reason to edit a threshold.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"
# Change to "main" once feat/m07b-freeze-d1 is merged.
REF = "feat/m07b-freeze-d1"

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Always sync to the target ref: a reused VM may hold a STALE clone.
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "fetch", "-q", "origin", REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "-q", "-B", REF, f"origin/{REF}"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{REF}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", REF, REPO_URL, REPO_DIR], check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit:", commit)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True
)

import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU."
free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb >= 10, (
    f"GPU has only {free_gb:.1f} GB free - dirty runtime. Runtime -> Disconnect and"
    " delete runtime, then reopen from GitHub and Run all."
)
print(f"GPU free: {free_gb:.1f} GB - OK ({torch.cuda.get_device_name(0)})")
ENV = dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src"), PROBE_REPO_COMMIT=commit)
CONFIG = "configs/m07b_freeze_d1.yaml"

In [ ]:
import json

# Read-only preflight: what is about to be frozen. Nothing is scored here.
SOURCE_DIR = os.path.join(DRIVE_ROOT, "d1-20261108-145253")
with open(os.path.join(SOURCE_DIR, "d1_report.json"), encoding="utf-8") as f:
    source = json.load(f)
print("source run:", source["run_id"], "| status:", source["status"])
print("best epoch:", source.get("best_epoch"), "| d1 sha256:", source.get("d1_model_sha256"))
print("training repo commit:", source.get("environment", {}).get("repo_commit"))
fe_report = os.path.join(SOURCE_DIR, "d1_final_eval_report.json")
print("final-eval already ran:", os.path.exists(fe_report), "(reused if so, never re-scored)")
frozen = os.path.join(DRIVE_ROOT, "d1-frozen-" + source["run_id"])
print("frozen dir exists:", os.path.exists(frozen), "(the run refuses to overwrite it)")
neg = os.path.join(REPO_DIR, "data/synthetic/short_eval_neg_v1.jsonl")
print("short negatives file present:", os.path.exists(neg))

## Run the freeze

Long step: final-eval, then the probe's timing pilot and generation.

In [ ]:
import glob

result = subprocess.run(
    [sys.executable, "-m", "pii_redteam.cli", "freeze-d1", "--config", CONFIG],
    cwd=REPO_DIR,
    env=ENV,
)
print(f"\nfreeze-d1 exit code: {result.returncode} (0 = frozen, 2 = freeze_blocked)")
if result.returncode not in (0, 2):
    print("^^ the traceback above is the real error - paste it back to me")
RUN_DIR = sorted(glob.glob(os.path.join(DRIVE_ROOT, "d1-freeze-2*")))[-1]
print("latest freeze run dir:", RUN_DIR)

In [ ]:
with open(os.path.join(RUN_DIR, "d1_freeze_report.json"), encoding="utf-8") as f:
    report = json.load(f)
print("STATUS:", report["status"])
for name, c in report["checks"].items():
    req = "" if c["required"] else " (optional)"
    print(f"  [{c['status'].upper()}] {name}{req}")
print("\nexperiments_entry (paste into EXPERIMENTS.yaml):")
print(json.dumps(report.get("experiments_entry"), indent=2))

**When the run finishes:** bring back `d1_freeze_report.json` from the freeze run
directory (and the probe's `m07a-d1-<timestamp>/probe_report.json`). The report's
`experiments_entry` goes into `EXPERIMENTS.yaml`.

**After a disconnect:** the final-eval report is kept in the training run directory and
is reused, not re-scored. If the probe finished (its directory has `probe_report.json`
and `samples.jsonl`), set `probe.reuse_run_dir` in `configs/m07b_freeze_d1.yaml` to that
directory to skip regenerating it; otherwise just Run all again.

**If blocked:** the failing checks are listed above. Do not change a threshold to get a
pass without a recorded owner decision.